# 02 — Data Preprocessing
Binarizes, filters, splits, and saves both datasets.
**Requires:** `01_data_downloading.ipynb` to have been run first.

In [11]:
import pandas as pd
import numpy as np
import pickle, gzip, json
from pathlib import Path
from typing import Tuple

ROOT = Path('..').resolve()  # project root (one level up from src/)
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


## Shared utility functions

In [12]:
def temporal_split(
    df: pd.DataFrame,
    user_col: str = 'user_id',
    time_col: str = 'timestamp',
    train_frac: float = 0.8,
    val_frac: float = 0.1,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Per-user chronological split: train / val / test.
    Avoids data leakage — train always contains older interactions than val/test.
    """
    df = df.sort_values([user_col, time_col])
    train_rows, val_rows, test_rows = [], [], []
    for _, group in df.groupby(user_col):
        n = len(group)
        if n < 3:
            continue
        i_train = max(int(n * train_frac), 1)
        i_val   = max(int(n * (train_frac + val_frac)), i_train + 1)
        if i_val >= n:
            continue
        train_rows.append(group.iloc[:i_train])
        val_rows.append(group.iloc[i_train:i_val])
        test_rows.append(group.iloc[i_val:])
    return (
        pd.concat(train_rows).reset_index(drop=True),
        pd.concat(val_rows).reset_index(drop=True),
        pd.concat(test_rows).reset_index(drop=True),
    )


def build_id_maps(df, user_col='user_id', item_col='item_id'):
    """Map raw ids to contiguous 0-based integers."""
    user2idx = {u: i for i, u in enumerate(sorted(df[user_col].unique()))}
    item2idx = {it: i for i, it in enumerate(sorted(df[item_col].unique()))}
    return user2idx, item2idx


def remap(df, user2idx, item2idx, user_col='user_id', item_col='item_id'):
    """Apply id maps, drop rows with unmapped ids."""
    df = df.copy()
    df[user_col] = df[user_col].map(user2idx)
    df[item_col] = df[item_col].map(item2idx)
    return df.dropna(subset=[user_col, item_col]).astype({user_col: int, item_col: int})


def kcore_filter(df, k=5, user_col='user_id', item_col='item_id'):
    """Iteratively remove users and items with fewer than k interactions."""
    prev_len, iteration = -1, 0
    while prev_len != len(df):
        prev_len = len(df)
        iteration += 1
        uc = df[user_col].value_counts()
        ic = df[item_col].value_counts()
        df = df[
            df[user_col].isin(uc[uc >= k].index) &
            df[item_col].isin(ic[ic >= k].index)
        ]
    print(f'  k-core ({k}) stable after {iteration} iterations: '
          f'{len(df):,} rows | {df[user_col].nunique():,} users | {df[item_col].nunique():,} items')
    return df


def flag_cold_start(train_df, test_df, user_col='user_id', min_interactions=5):
    """Add cold_start boolean column to test set."""
    counts = train_df[user_col].value_counts()
    cold_users = set(counts[counts < min_interactions].index)
    test_df = test_df.copy()
    test_df['cold_start'] = test_df[user_col].isin(cold_users)
    return test_df


def save_dataset(name, train, val, test, user2idx, item2idx):
    """Save processed splits and metadata to data/processed/<name>/."""
    out = PROC / name
    out.mkdir(exist_ok=True)
    train.to_parquet(out / 'train.parquet')
    val.to_parquet(out   / 'val.parquet')
    test.to_parquet(out  / 'test.parquet')
    with open(out / 'meta.pkl', 'wb') as f:
        pickle.dump({
            'user2idx': user2idx, 'item2idx': item2idx,
            'n_users': len(user2idx), 'n_items': len(item2idx),
        }, f)
    print(f'  Saved to {out}/')


print('Utility functions defined.')

Utility functions defined.


## MovieLens 100K (explicit feedback)

In [13]:
print('=== MovieLens 100K ===')

# Step 1: Load
ml_raw = pd.read_csv(
    RAW / 'ml-100k' / 'u.data', sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp'],
)
print(f'Raw: {len(ml_raw):,} ratings | {ml_raw.user_id.nunique():,} users | {ml_raw.item_id.nunique():,} items')
ml_raw.head(3)

=== MovieLens 100K ===
Raw: 100,000 ratings | 943 users | 1,682 items


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116


In [14]:
# Step 2: Binarization — keep ratings >= 4 as positive (label=1), drop the rest
# Ratings < 4 are ambiguous and excluded to keep training signal clean
RATING_THRESHOLD = 4.0

ml_pos = ml_raw[ml_raw['rating'] >= RATING_THRESHOLD].copy()
ml_pos['label'] = 1
print(f'After binarization (>={RATING_THRESHOLD}): {len(ml_pos):,} positive interactions')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ml_raw['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.axvline(x=RATING_THRESHOLD - 1 - 0.5, color='red', linestyle='--', label=f'threshold={RATING_THRESHOLD}')
ax.set_title('MovieLens rating distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(PROC / 'ml_rating_dist.png', dpi=100)
plt.show()
print('Distribution plot saved.')

After binarization (>=4.0): 55,375 positive interactions
Distribution plot saved.


/var/folders/ys/p07ql4sx001fklpgvlls_rjm0000gp/T/ipykernel_15666/1674636175.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Step 3: k-core filtering
ml_filt = kcore_filter(ml_pos, k=5)

  k-core (5) stable after 2 iterations: 54,413 rows | 938 users | 1,008 items


In [16]:
# Step 4: ID remapping
ml_user2idx, ml_item2idx = build_id_maps(ml_filt)
ml_filt = remap(ml_filt, ml_user2idx, ml_item2idx)
n_users_ml, n_items_ml = len(ml_user2idx), len(ml_item2idx)
print(f'n_users={n_users_ml:,}  n_items={n_items_ml:,}')

n_users=938  n_items=1,008


In [17]:
# Step 5: Temporal split
ml_train, ml_val, ml_test = temporal_split(ml_filt)

# Step 6: Cold-start flagging
ml_test = flag_cold_start(ml_train, ml_test)

total = len(ml_train) + len(ml_val) + len(ml_test)
print(f'Train: {len(ml_train):,} ({len(ml_train)/total:.1%})')
print(f'Val:   {len(ml_val):,} ({len(ml_val)/total:.1%})')
print(f'Test:  {len(ml_test):,} ({len(ml_test)/total:.1%})')
print(f'Cold-start in test: {ml_test["cold_start"].mean():.1%}')

Train: 43,142 (79.3%)
Val:   5,398 (9.9%)
Test:  5,853 (10.8%)
Cold-start in test: 0.1%


In [18]:
# Save
save_dataset('movielens_100k', ml_train, ml_val, ml_test, ml_user2idx, ml_item2idx)
print('MovieLens 100K preprocessing complete.')

  Saved to /Users/XuanNguyen/Documents/NUS/DSS5104/project2_rcmsys/data/processed/movielens_1m/
MovieLens preprocessing complete.


## LastFM-2K (implicit feedback)

In [20]:
print('=== LastFM-2K ===')

# Step 1: Load — columns: userID, artistID, weight (listening count)
lastfm_raw = pd.read_csv(
    RAW / 'lastfm-2K' / 'user_artists.dat', sep='\t',
).rename(columns={'userID': 'user_id', 'artistID': 'item_id', 'weight': 'timestamp'})
# 'weight' (play count) used as proxy timestamp for temporal ordering
print(f'Raw: {len(lastfm_raw):,} interactions | {lastfm_raw.user_id.nunique():,} users | {lastfm_raw.item_id.nunique():,} items')
lastfm_raw.head(3)

=== LastFM-2K ===
Raw: 92,834 interactions | 1,892 users | 17,632 items


,user_id,item_id,timestamp
0,2,51,13883
1,2,52,11690
2,2,53,11351


In [21]:
# Step 2: Already deduplicated (one row per user-artist pair)
lastfm_dedup = lastfm_raw.copy()
print(f'Interactions: {len(lastfm_dedup):,}')

Interactions: 92,834


In [22]:
# Step 3: k-core filtering
lastfm_filt = kcore_filter(lastfm_dedup, k=5)

  k-core (5) stable after 4 iterations: 71,355 rows | 1,859 users | 2,823 items


In [23]:
# Step 4: ID remapping
lastfm_user2idx, lastfm_item2idx = build_id_maps(lastfm_filt)
lastfm_filt = remap(lastfm_filt, lastfm_user2idx, lastfm_item2idx)
n_users_lastfm, n_items_lastfm = len(lastfm_user2idx), len(lastfm_item2idx)
print(f'n_users={n_users_lastfm:,}  n_items={n_items_lastfm:,}')

n_users=1,859  n_items=2,823


In [24]:
# Steps 5 & 6: Temporal split + cold-start flagging
lastfm_train, lastfm_val, lastfm_test = temporal_split(lastfm_filt)
lastfm_test = flag_cold_start(lastfm_train, lastfm_test)

total = len(lastfm_train) + len(lastfm_val) + len(lastfm_test)
print(f'Train: {len(lastfm_train):,} ({len(lastfm_train)/total:.1%})')
print(f'Val:   {len(lastfm_val):,} ({len(lastfm_val)/total:.1%})')
print(f'Test:  {len(lastfm_test):,} ({len(lastfm_test)/total:.1%})')
print(f'Cold-start in test: {lastfm_test["cold_start"].mean():.1%}')

Train: 56,343 (79.0%)
Val:   7,093 (9.9%)
Test:  7,899 (11.1%)
Cold-start in test: 0.1%


In [25]:
# Save
save_dataset('lastfm_2k', lastfm_train, lastfm_val, lastfm_test, lastfm_user2idx, lastfm_item2idx)
print('LastFM-2K preprocessing complete.')
print('\nAll datasets ready. Run 03_baseline.ipynb next.')

  Saved to /Users/XuanNguyen/Documents/NUS/DSS5104/project2_rcmsys/data/processed/lastfm_2k/
LastFM-2K preprocessing complete.

All datasets ready. Run 03_baseline.ipynb next.
